# ENN583 Week 11 — Grasp Prediction with GG-CNN

This practical uses the public [GG-CNN implementation](https://github.com/dougsm/ggcnn) to predict planar parallel-jaw grasps from a depth image.

We will:

1. load a paired RGB and depth image;
2. crop and prepare the depth data for GG-CNN;
3. interpret the predicted grasp quality, angle, and width maps; and
4. extract and visualise a rectangle grasp in the original image.

## 1. Setup

The pretrained Cornell checkpoint and GG-CNN model definition are included with this practical. The code will use CUDA when it is available and otherwise run on CPU.

In [ ]:
from pathlib import Path
import math

import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from scipy import ndimage
import torch

from models.ggcnn2 import GGCNN2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using: {device}')
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))

## 2. Load an RGB-D example

The RGB and depth images have matching filenames. We begin with scene `0004`, which contains several separated objects. Scene `0000` is more cluttered and will be used later.

The depth PNG must be read with `IMREAD_UNCHANGED`. This preserves its 16-bit millimetre measurements; loading it as an ordinary grayscale image would discard depth precision.

In [ ]:
sample_id = '0004'
image_dir = Path('grasping_images')
rgb_path = image_dir / 'rgb' / f'{sample_id}.png'
depth_path = image_dir / 'depth' / f'{sample_id}.png'

im_rgb = cv2.cvtColor(cv2.imread(str(rgb_path)), cv2.COLOR_BGR2RGB)
im_depth_mm = cv2.imread(str(depth_path), cv2.IMREAD_UNCHANGED)

if im_rgb is None or im_depth_mm is None:
    raise FileNotFoundError('Could not load the RGB-depth image pair')

print('RGB:', im_rgb.shape, im_rgb.dtype)
print('Depth:', im_depth_mm.shape, im_depth_mm.dtype)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(im_rgb)
axes[0].set_title('RGB image')
axes[0].axis('off')
depth_plot = axes[1].imshow(im_depth_mm, cmap='viridis')
axes[1].set_title('Depth image (mm)')
axes[1].axis('off')
fig.colorbar(depth_plot, ax=axes[1], fraction=0.046)
plt.show()

## 3. Crop around a target object

GG-CNN accepts a `300 × 300` depth image. It is useful to give it a square crop around the object we want to grasp. Keeping the crop square avoids distorting grasp positions and angles when it is resized.

### Your turn: crop around the banana

Adjust `x_offset`, `y_offset`, and `crop_size` so the crop is centred around the banana. The default centre crop is valid, so the rest of the notebook still runs before you refine it.

**Consider:** How could a detector or instance-segmentation model automate this crop?

In [ ]:
# A valid square centre crop to start from.
crop_size = min(im_rgb.shape[:2])
x_offset = (im_rgb.shape[1] - crop_size) // 2
y_offset = (im_rgb.shape[0] - crop_size) // 2

# Find a tighter crop around the banana in scene 0004. Feel free to use an object detector or semantic segmenation model here!
# TODO: Implement this solution.
pass

rgb_crop_original = im_rgb[
    y_offset:y_offset + crop_size, x_offset:x_offset + crop_size
]
depth_crop_original = im_depth_mm[
    y_offset:y_offset + crop_size, x_offset:x_offset + crop_size
]

crop_rgb = cv2.resize(rgb_crop_original, (300, 300), interpolation=cv2.INTER_AREA)
crop_depth_mm = cv2.resize(
    depth_crop_original, (300, 300), interpolation=cv2.INTER_NEAREST
)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(crop_rgb)
axes[0].set_title('RGB crop')
axes[0].axis('off')
depth_crop_plot = axes[1].imshow(crop_depth_mm, cmap='viridis')
axes[1].set_title('Depth crop (mm)')
axes[1].axis('off')
fig.colorbar(depth_crop_plot, ax=axes[1], fraction=0.046)
plt.show()

## 4. Prepare the depth image

A depth camera can return zero where no valid measurement was obtained. The function below identifies those pixels, expands the invalid region slightly, and fills it using neighbouring depth values. It then converts millimetres to metres.

In [ ]:
def clean_depth_image(depth_mm, out_size=300):
    depth = depth_mm.astype(np.float32) / 1000.0
    depth = cv2.copyMakeBorder(depth, 1, 1, 1, 1, cv2.BORDER_REPLICATE)

    invalid_mask = (~np.isfinite(depth) | (depth <= 0)).astype(np.uint8)
    invalid_mask = cv2.dilate(invalid_mask, np.ones((3, 3), np.uint8), iterations=1)
    depth[invalid_mask == 1] = 0

    depth_scale = np.abs(depth).max()
    if depth_scale == 0:
        raise ValueError('The depth crop contains no valid measurements')

    depth_scaled = depth / depth_scale
    depth_scaled = cv2.inpaint(depth_scaled, invalid_mask, 1, cv2.INPAINT_NS)
    depth = depth_scaled[1:-1, 1:-1] * depth_scale
    depth = cv2.resize(depth, (out_size, out_size), interpolation=cv2.INTER_AREA)
    return depth

clean_depth = clean_depth_image(crop_depth_mm)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(crop_depth_mm, cmap='viridis')
axes[0].set_title('Raw depth crop (mm)')
axes[1].imshow(clean_depth, cmap='viridis')
axes[1].set_title('Clean depth crop (m)')
for ax in axes:
    ax.axis('off')
plt.show()

GG-CNN was trained using mean-centred depth crops. We subtract the crop mean, clip extreme differences, add batch and channel dimensions, and move the tensor to the selected device.

In [ ]:
depth_normalised = np.clip(clean_depth - clean_depth.mean(), -1, 1)
depth_tensor = torch.from_numpy(
    depth_normalised.reshape(1, 1, 300, 300).astype(np.float32)
).to(device)

print('Input tensor:', tuple(depth_tensor.shape), depth_tensor.dtype, depth_tensor.device)

## 5. Predict dense grasp maps

GG-CNN predicts four dense maps:

- grasp quality at every pixel;
- cosine and sine components representing grasp orientation; and
- gripper width.

The angle is recovered with `atan2(sin, cos) / 2` because the network represents twice the grasp angle. A parallel-jaw grasp is unchanged by a rotation of π radians.

In [ ]:
model = GGCNN2()
state_dict = torch.load(
    'ggcnn2_weights_cornell/epoch_50_cornell_statedict.pt',
    map_location=device,
    weights_only=True,
)
model.load_state_dict(state_dict)
model = model.to(device).eval()

with torch.inference_mode():
    quality, cos, sin, width = model(depth_tensor)

quality = quality.squeeze().cpu().numpy()
cos = cos.squeeze().cpu().numpy()
sin = sin.squeeze().cpu().numpy()
width = width.squeeze().cpu().numpy()
angle = np.arctan2(sin, cos) / 2.0

print('Output map shape:', quality.shape)

### Your turn: inspect the raw maps

Visualise the RGB crop alongside the raw quality, angle, and width maps. Where does the model predict high-quality grasps? Do the orientations agree with the objects you see?

In [ ]:
# TODO: Implement this solution.
pass


## 6. Post-process the grasp maps

Following the GG-CNN implementation, the width map is converted to pixels and the maps are smoothed. Smoothing suppresses isolated noisy responses and favours coherent grasp regions.

In [ ]:
def postprocess_grasp_maps(quality_map, angle_map, width_map):
    quality_map = ndimage.gaussian_filter(quality_map, 2.0)
    angle_map = ndimage.gaussian_filter(angle_map, 2.0)
    width_map = ndimage.gaussian_filter(width_map * 150.0, 1.0)
    quality_map = np.clip(quality_map, 0, 1.0 - 1e-3)
    return quality_map, angle_map, width_map

quality_final, angle_final, width_final = postprocess_grasp_maps(
    quality, angle, width
)

### Your turn: compare before and after post-processing

Visualise the post-processed maps. How did smoothing change the strongest grasp regions?

In [ ]:
# TODO: Implement this solution.
pass


## 7. Extract and draw the best grasp

We will take the highest-quality pixel as the grasp centre, then read its predicted angle and gripper width. This is a simple selection strategy: a deployed system would normally consider multiple local maxima, object masks, reachability, and collision checks.

In [ ]:
def rotated_rectangle_corners(center_x, center_y, width, angle_rad, height=30):
    half_width = width / 2
    half_height = height / 2
    corners = [
        (-half_width, -half_height),
        (half_width, -half_height),
        (half_width, half_height),
        (-half_width, half_height),
    ]

    return [
        (
            center_x + x * math.cos(angle_rad) - y * math.sin(angle_rad),
            center_y + x * math.sin(angle_rad) + y * math.cos(angle_rad),
        )
        for x, y in corners
    ]

v, u = np.unravel_index(np.argmax(quality_final), quality_final.shape)
best_angle = angle_final[v, u]
best_width = width_final[v, u]
grasp_crop = rotated_rectangle_corners(u, v, best_width, best_angle)

print(f'Centre: ({u}, {v})')
print(f'Quality: {quality_final[v, u]:.3f}')
print(f'Angle: {np.degrees(best_angle):.1f} degrees')
print(f'Width: {best_width:.1f} pixels')

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(crop_rgb)
ax.add_patch(patches.Polygon(
    grasp_crop, closed=True, edgecolor='red', linewidth=2, fill=False
))
ax.set_title('Best grasp in the 300 × 300 crop')
ax.axis('off')
plt.show()

The grasp coordinates above belong to the resized crop. Because the source crop was square, the same scale applies horizontally and vertically. We transform every rectangle corner back to the original image instead of separately reconstructing its width and height.

In [ ]:
crop_scale = crop_size / 300.0
grasp_global = [
    (x * crop_scale + x_offset, y * crop_scale + y_offset)
    for x, y in grasp_crop
]

fig, ax = plt.subplots(figsize=(12, 7))
ax.imshow(im_rgb)
ax.add_patch(patches.Polygon(
    grasp_global, closed=True, edgecolor='red', linewidth=3, fill=False
))
ax.set_title('Predicted grasp in the original image')
ax.axis('off')
plt.show()

## 8. Your turn: investigate the two scenes

1. Change `sample_id` to `0000` and rerun from the image-loading cell.
2. Choose a square crop around an object you want to grasp.
3. Compare the raw and post-processed maps.
4. Check whether the most confident rectangle is physically plausible.
5. Try a broader crop containing several objects. Does the globally best grasp belong to your intended object?

Discuss:

- Why does GG-CNN need depth but not RGB?
- How could the Mask R-CNN or Mask2Former results from the earlier practical restrict grasp selection to one object?
- What additional information is required to turn this image-space rectangle into a robot command?
- What limitations arise because GG-CNN predicts planar grasps rather than full 6-DoF gripper poses?